In [1]:
import glob
import os
from pathlib import Path
import json
video_paths = glob.glob(os.path.join("/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie", "**/*.mp4"), recursive=True)
y = []
for path in Path(r"/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie").rglob('*.json'):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        y.append(data['emotion'])
        
print(set(y))

{'fear', 'sadness', 'contentment', 'awe', 'excitement', 'amusement', 'anger', 'disgust'}


In [32]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(video_paths, y, train_size = 0.8, random_state = 42, stratify = y)
print(len(X_train))

132


In [33]:
video_paths[:5], y[:5]

(['/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_017.mp4',
  '/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_006.mp4',
  '/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_032.mp4',
  '/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_007.mp4',
  '/kaggle/input/datasets/timurbartia/emovid-private/EmoVid_Data/movie/video/00114/00114_clip_036.mp4'],
 ['awe', 'awe', 'excitement', 'excitement', 'awe'])

In [34]:
from transformers import VJEPA2Model, AutoVideoProcessor, TrainingArguments, Trainer
import torch
from datasets import Dataset

video_processor = AutoVideoProcessor.from_pretrained("facebook/vjepa2-vitl-fpc16-256-ssv2")
model = VJEPA2Model.from_pretrained("facebook/vjepa2-vitl-fpc16-256-ssv2").to('cuda')

train_dataset = Dataset.from_dict({
    "video_path": X_train,
    "label": y_train
})
train_dataset = train_dataset.class_encode_column("label")

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

VJEPA2Model LOAD REPORT from: facebook/vjepa2-vitl-fpc16-256-ssv2
Key                                                              | Status     |  | 
-----------------------------------------------------------------+------------+--+-
pooler.self_attention_layers.{0, 1, 2}.layer_norm2.bias          | UNEXPECTED |  | 
pooler.cross_attention_layer.mlp.fc2.bias                        | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.q_proj.bias     | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.v_proj.weight   | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.v_proj.bias     | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.out_proj.bias   | UNEXPECTED |  | 
pooler.cross_attention_layer.layer_norm1.bias                    | UNEXPECTED |  | 
pooler.cross_attention_layer.mlp.fc1.bias                        | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.layer_norm2.weight        | UNEXPECTED |  | 
pooler.sel

Casting to class labels:   0%|          | 0/132 [00:00<?, ? examples/s]

In [35]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

def video_preprocessing(batch):
  X = []
  y = []

  video_paths = batch['video_path']
  labels = batch['label']

  for path, label in zip(video_paths, labels):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
      continue

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < 16:
      cap.release()
      continue

    idxs = np.linspace(0, total_frames - 1, 16, dtype = int)
    frames = []

    for f_idx in idxs:
      cap.set(cv2.CAP_PROP_POS_FRAMES, f_idx)
      ret, frame = cap.read()
      if not ret:
        break
      frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
      frames.append(frame)
    cap.release()

    if len(frames) == 16:
      proc_frames = video_processor(videos = frames, return_tensors = 'pt')
      key = list(proc_frames.keys())[0]
      X.append(proc_frames[key].squeeze(0))
      y.append(label)
    
  if not X:
    return {"vjepa_embedding": [], "label": []}

  inputs = torch.stack(X).to('cuda')
  with torch.no_grad():
    outputs = model(inputs)
    embeddings = torch.mean(outputs.last_hidden_state, dim = 1)

  return {
    "vjepa_embedding": embeddings.cpu().numpy().tolist(),
    "label": y
  }


In [36]:
emb_dataset = train_dataset.map(
    video_preprocessing,
    batched = True,
    batch_size = 32,
    remove_columns = ['video_path']
)

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

In [42]:
print(len(emb_dataset['vjepa_embedding'][0]))
emb_dataset.save_to_disk("132_video_embeddings_dataset")

1024


Saving the dataset (0/1 shards):   0%|          | 0/132 [00:00<?, ? examples/s]

In [57]:
test_dataset = Dataset.from_dict({
    "video_path": X_test,
    "label": y_test
})
label_feature = train_dataset.features["label"]

test_dataset = test_dataset.cast_column("label", label_feature)

Casting the dataset:   0%|          | 0/2651 [00:00<?, ? examples/s]

In [58]:
test_results = trainer.predict(test_dataset=test_dataset.select(range(100)))
test_results.label_ids

array([3, 1, 4, 0, 6, 1, 4, 0, 6, 4, 6, 5, 0, 5, 2, 5, 2, 6, 5, 2, 1, 4,
       7, 6, 5, 6, 7, 5, 2, 5, 7, 5, 0, 5, 7, 3, 4, 3, 4, 2, 1, 6, 1, 1,
       4, 4, 6, 7, 6, 5, 3, 3, 3, 1, 3, 4, 1, 6, 1, 3, 0, 2, 0, 2, 6, 0,
       1, 7, 1, 4, 5, 1, 1, 4, 7, 0, 0, 6, 4, 1, 5, 7, 2, 1, 3, 7, 1, 5,
       0, 5, 0, 0, 0, 5, 7, 1, 6, 6, 4, 1])

In [59]:
from sklearn.metrics import classification_report
pred = np.argmax(test_results.predictions, axis=-1)
print(classification_report(test_results.label_ids, pred))

              precision    recall  f1-score   support

           0       0.22      0.15      0.18        13
           1       0.30      0.50      0.38        18
           2       0.60      0.38      0.46         8
           3       0.00      0.00      0.00         9
           4       0.33      0.23      0.27        13
           5       0.06      0.07      0.06        15
           6       0.42      0.57      0.48        14
           7       0.43      0.30      0.35        10

    accuracy                           0.29       100
   macro avg       0.30      0.27      0.27       100
weighted avg       0.28      0.29      0.28       100

